# BACI

## b) Creazione matrici Mcp annuali (a partire dal file parquet complessivo )

In [ ]:
import pandas as pd
import sys
from pathlib import Path
ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

from src.pipeline import Mcp_pipeline
from src.filters import calculate_rca
from src.pipeline import build_Mcp_matrix

In [ ]:



Mcp_pipeline(
    df,
    country_col ='j',
    product_col = 'k',
    value_col = 'v',
    year_col = 't',
    output_dir = r"C:\Users\vitto\Desktop\CSH RESEARCH\data\HS22_2022_4_digits\Mcp\Mcp_import"
)



### Prodotti in comune Mcp export ed Mcp import

In [1]:
import numpy as np
import pandas as pd
import sys
import os
from pathlib import Path
import networkx as nx
import matplotlib.pyplot as plt


ROOT = Path().resolve().parent  

sys.path.insert(0, str(ROOT))


from src.preprocessing import M_and_C_common_products


In [2]:
M_E = pd.read_parquet(r"C:\Users\vitto\Desktop\CSH RESEARCH\data\HS22_2022_4_digits\Mcp\Mcp_export\Mcp_export_2022_filtered.parquet")

In [3]:
M_I = pd.read_parquet(r"C:\Users\vitto\Desktop\CSH RESEARCH\data\HS22_2022_4_digits\Mcp\Mcp_import\Mcp_import_2022.parquet")

In [4]:
missing_1,missing_2 = M_and_C_common_products(M_I,M_E)

Numero di colonne Mcp:  1228
Numero di righe Cpp':  1189
prodotti in comune tra righe di Cpp e Mcp: 1189
Prodotti presenti in M ma non in C: {'9606', '9616', '9506', '9404', '9620', '9406', '9701', '9615', '9611', '9507', '9504', '9608', '9702', '9609', '9612', '9401', '9604', '9617', '9601', '9704', '9402', '9603', '9610', '9705', '9505', '9607', '9503', '9706', '9403', '0508', '9703', '9613', '9405', '9614', '9619', '9605', '9618', '9602', '9508'}
Numero di Prodotti presenti in M ma non in C: 39
Numero di Prodotti presenti in M ma non in C: set()
Prodotti presenti in C ma non in M: 0


## Creazione MATRICE Mcp senza codici non congruenti con AIPNET

In [7]:
import pandas as pd
import sys
from pathlib import Path
ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

from src.pipeline import Mcp_pipeline
from src.filters import calculate_rca
from src.pipeline import build_Mcp_matrix

In [8]:
# Carica il database BACI
df = pd.read_parquet(r"C:\Users\vitto\Desktop\CSH RESEARCH\data\HS22_2022_4_digits\BACI\BACI_IMPORT_4_digits_HS22_all_years.parquet")

# Filtra solo il 2022
M_I = df[df["t"] == 2022]

# Rimuovi i codici k in missing_1
df_filtered = M_I[~M_I["k"].isin(missing_1)]

# Crea la matrice Mcp
Mcp_pipeline(
    df_filtered,
    country_col='j',
    product_col='k',
    value_col='v',
    year_col='t',
    output_dir=r"C:\Users\vitto\Desktop\CSH RESEARCH\data\HS22_2022_4_digits\Mcp\Mcp_import"
)

{2022: k    0101  0102  0103  0104  0105  0106  0201  0202  0203  0204  ...  9207  \
 j                                                                ...         
 4     1.0   0.0   0.0   0.0   1.0   0.0   0.0   0.0   0.0   0.0  ...   0.0   
 8     0.0   1.0   1.0   1.0   1.0   1.0   0.0   1.0   1.0   0.0  ...   0.0   
 12    0.0   1.0   0.0   0.0   1.0   0.0   0.0   0.0   0.0   0.0  ...   0.0   
 16    0.0   0.0   0.0   0.0   0.0   0.0   0.0   1.0   0.0   1.0  ...   0.0   
 20    0.0   0.0   0.0   0.0   0.0   0.0   1.0   0.0   1.0   1.0  ...   1.0   
 ..    ...   ...   ...   ...   ...   ...   ...   ...   ...   ...  ...   ...   
 862   0.0   0.0   0.0   0.0   1.0   1.0   0.0   0.0   0.0   0.0  ...   0.0   
 876   0.0   0.0   0.0   0.0   0.0   0.0   1.0   1.0   1.0   1.0  ...   0.0   
 882   0.0   0.0   0.0   0.0   1.0   0.0   0.0   1.0   1.0   1.0  ...   1.0   
 887   0.0   0.0   0.0   1.0   1.0   1.0   0.0   0.0   0.0   0.0  ...   0.0   
 894   0.0   0.0   0.0   0.0   1.0   0.0   0.0

## Aggiunta colonna descrizione prodotto HS4

### Creazione legenda codici HS4 

Prima avevo solo la legenda a livello HS6

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv(r"C:\Users\vitto\Desktop\CSH RESEARCH\data\raw\BACI\BACI_HS22\LEGENDE\product_codes_HS22_V202601.csv")

In [ ]:
# il file con i codici HS6 ha colonne "code"| "description"

# estraggo le prime 4 cifre del codice HS6
# prima converto in stringa e aggiungo zeri dove necessario
df["code"] = df["code"].astype(str).str.zfill(6) 
df['code'] = df['code'].astype(str).str[:-2]


# funzione per prendere solo il testo prima dei :
def clean_desc(desc):
    if ':' in desc:
        return desc.split(':')[0] # prendo la parte a sx dei : 
    return desc

df['description'] = df['description'].apply(clean_desc)

# prendi la descrizione più frequente per ogni HS4
df_hs4 = (
    df.groupby('code')['description']
    .agg(lambda x: x.value_counts().idxmax())
    .reset_index())
df_hs4.to_csv(r"C:\Users\vitto\Desktop\CSH RESEARCH\data\raw\BACI\BACI_HS22\LEGENDE\product_codes_HS22_4_V202601.csv", index= False)


# AIPNET

### AIPNET ---> C

In [ ]:
import numpy as np
import pandas as pd
import sys
import os
from pathlib import Path

ROOT = Path().resolve().parent  
sys.path.insert(0, str(ROOT))

from src.preprocessing import AIPNET_to_C

In [ ]:
df = pd.read_csv(r"C:\Users\vitto\Desktop\CSH RESEARCH\data\raw\AIPNET\AIPNET_4_digits_HS22.csv")
print(df.columns)
upstream_product_column_name = "hs2022_code_upstream"
downstream_product_column_name = "hs2022_code_downstream"
N = 4

In [ ]:
C = AIPNET_to_C(df, upstream_product_column_name, downstream_product_column_name, N)

### Verifica proprietà

In [ ]:
count_0s = np.sum(C == 0)
count_1s = np.sum(C == 1)
print(count_0s, count_1s)

### Salvataggio

In [ ]:
C.to_parquet(r"C:\Users\vitto\Desktop\CSH RESEARCH\data\processed\AIPNET\C_4_digits_HS22.parquet")

### Colonne in comune/diverse BACI-AIPNET

In [ ]:
import pandas as pd
import sys
from pathlib import Path

ROOT = Path().resolve().parent 
sys.path.insert(0, str(ROOT))

from src.preprocessing import M_and_C_common_products

In [ ]:
M = pd.read_parquet(r"C:\Users\vitto\Desktop\CSH RESEARCH\data\HS22_2022_4_digits\Mcp\Mcp_export\Mcp_export_2022.parquet")
C = pd.read_parquet(r"C:\Users\vitto\Desktop\CSH RESEARCH\data\HS22_2022_4_digits\AIPNET\C_4_digits_HS22.parquet")

In [ ]:
M_and_C_common_products(M, C)

## Nilpotenza e autovettore dominante matrice C

## Nilpotenza

### Verifica booleana

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import glob

In [ ]:
def nihilpotency(C, tol=1e-10):
    """
    Verifica se una matrice è nilpotente usando gli autovalori.

    Input:
    
    -C : np.ndarray
        Matrice quadrata
    -tol : float
        Tolleranza numerica

    Output:
    
    -variabile booleana: 
    True : tutti gli autovalori sono nulli
    False : altrimenti
    """
    
    # una matrice è nilpotente se tutti gli autovalori sono nulli
    eigenvalues = np.linalg.eigvals(C)

    return eigenvalues[1], np.all(np.abs(eigenvalues) < tol)

In [ ]:
# cartella con i csv
cartella_AIPNET_raw = Path(r"C:\Users\vitto\Desktop\CSH RESEARCH\data\processed\AIPNET")

# runno la funzione per tutti i file csv dela cartella
for file in cartella_AIPNET_raw.glob("*.parquet"):   # glob() : cerca file / cartelle che corrispondono alla descrizione 
    
    df = pd.read_parquet(file)
    
    # esegui funzione
    result = nihilpotency(df, 1e-10)


    print (result)

Probabilmente c'è qualche problema nella matrice C HS22 6-digits

### TEST A) Verifica che il numero di link delle reti coincida con quello dichiarato da AIPNET

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import glob

In [ ]:
# cartella con i csv
cartella_AIPNET_raw = Path(r"C:\Users\vitto\Desktop\CSH RESEARCH\data\processed\AIPNET")

# runno la funzione per tutti i file csv dela cartella
for file in cartella_AIPNET_raw.glob("*.parquet"):   # glob() : cerca file / cartelle che corrispondono alla descrizione 
    
    df = pd.read_parquet(file)
    
    # esegui funzione
    count_0s, count_1s = count_number_of_0s_and_1s(df)

    print("number of 0s: ", count_0s)
    print("number of 1s: ", count_1s)

In [ ]:
def C_test(df):
    
    """
    Verifica che la matrice C abbia correttamente gli stessi codici su righe e colonne
    Se tutto è corretto dovrebbe dare 0

    Input:
    - Matrice C

    Output:
    - Numero di codici diversi tra righe e colonne
    
    """

    # Codici diversi riga e colonna
    C_codes_row = len(set(pd.unique(C.index.astype(str))))
    C_codes_columns = len(set(pd.unique(C.columns.astype(str))))
    missing_total =  C_codes_row ^ C_codes_columns

    return C_codes_row, C_codes_columns, missing_total,  

In [ ]:
C = pd.read_parquet(r"C:\Users\vitto\Desktop\CSH RESEARCH\data\processed\AIPNET\C_4_digits_HS22.parquet")
print (C)

In [ ]:
C_test(C)

In [ ]:
# LETTURA FILES

# file con i risultati di total_complexity (Q) 
df_esterno = pd.read_parquet(r"C:\Users\vitto\Desktop\CSH RESEARCH\results\real_data\2022_hs22_4_digits\products.parquet")
total_complexity = df_esterno["total_complexity"]

# indice di ogni componente di total_complexity (Q)
df_esterno = df_esterno.set_index("product")

In [ ]:
import matplotlib.pyplot as plt

In [ ]:

    # 2. Allineo gli indici (importantissimo)
    df_allineato = pd.concat([autovettore_principale, total_complexity], axis=1, join='inner')
    df_allineato.columns = ["autovettore", "total_complexity"]
    
    print(df_allineato)

    # 4. Calcola correlazione
    correlazione = df_allineato.corr().iloc[0,1]
    print("Correlazione:", correlazione)
    
    # 5. Grafico scatter
    plt.figure()
    plt.scatter(df_allineato["autovettore"], df_allineato["total_complexity"])
    plt.xlabel("Autovettore principale")
    plt.ylabel("total complexity")
    plt.title(f"Correlazione = {correlazione:.3f}")
    plt.grid()
    plt.show()
   